# Multi-Digit Handwritten String Recognition (Kaggle Notebook)

This notebook builds a **complete training and inference pipeline** for recognizing variable-length handwritten digit strings (e.g., `"123"`, `"45678"`) from synthetic images generated from Kaggle's Digit Recognizer dataset.

We compare two approaches:
1. **Segmentation + single-digit CNN classifier** (classical pipeline).
2. **YOLOv8 object detector** for end-to-end digit localization + classification (preferred when digits touch).

## Why this design?

- Kaggle Digit Recognizer gives isolated digits only. To solve sequence recognition, we **synthesize multi-digit images** with controllable spacing and overlap.
- Touching digits are difficult for simple contour segmentation, so we include both:
  - an interpretable OpenCV-based splitter,
  - and a robust detector (YOLOv8) trained directly on crowded strings.
- We evaluate with:
  - **Sequence accuracy** (entire string correct),
  - **Per-digit classification accuracy**,
  - and detection metrics for YOLO.

In [ ]:
# =========================
# 1) Setup
# =========================
import os
import random
import math
import shutil
from pathlib import Path
from dataclasses import dataclass
from typing import List, Tuple, Dict

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import torch  # still useful for YOLO device checks
import albumentations as A
from sklearn.model_selection import train_test_split

sns.set_style("whitegrid")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("TensorFlow:", tf.__version__)
print("Torch device for YOLO:", DEVICE)

In [ ]:
# Optional: install ultralytics in Kaggle if not already present.
# Uncomment when running first time.
# !pip install -q ultralytics

## 2) Load MNIST directly from TensorFlow/Keras

Per your requirement, we use `tensorflow.keras.datasets.mnist` directly instead of CSV files.

- Train split: 60,000 digits
- Test split: 10,000 digits
- Each image: 28x28 grayscale

In [ ]:
# Load MNIST from TensorFlow datasets
(x_train_mnist, y_train_mnist), (x_test_mnist, y_test_mnist) = keras.datasets.mnist.load_data()

images = np.concatenate([x_train_mnist, x_test_mnist], axis=0).astype(np.uint8)
labels = np.concatenate([y_train_mnist, y_test_mnist], axis=0).astype(np.int64)

print('Images:', images.shape, 'Labels:', labels.shape)
print('Label distribution (first few):', np.bincount(labels)[:10])

In [ ]:
plt.figure(figsize=(10, 3))
for i in range(12):
    plt.subplot(2, 6, i + 1)
    idx = np.random.randint(0, len(images))
    plt.imshow(images[idx], cmap='gray')
    plt.title(str(labels[idx]))
    plt.axis('off')
plt.tight_layout()
plt.show()

## 3) Synthetic multi-digit generator (2-5 digits)

We create variable-length string images by composing MNIST-style digit crops horizontally with:
- random gaps (`0..5` px),
- optional overlap (up to `10%` digit width),
- geometric + photometric augmentation,
- additive noise.

Each synthetic sample stores:
- full image,
- sequence label string,
- per-digit bounding boxes.

In [ ]:
# Group source digits by class for balanced sampling.
by_digit = {d: images[labels == d] for d in range(10)}
for d in range(10):
    print(f"digit {d}: {len(by_digit[d])}")

geom_aug = A.Compose([
    A.Affine(scale=(0.85, 1.15), rotate=(-15, 15), shear=(-12, 12), p=0.9),
    A.ElasticTransform(alpha=8, sigma=5, p=0.25),
], additional_targets={})

photo_aug = A.Compose([
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.GaussNoise(std_range=(0.01, 0.06), p=0.4),
    A.CoarseDropout(num_holes_range=(1, 4), hole_height_range=(2, 6), hole_width_range=(2, 6), fill=0, p=0.3),
])

@dataclass
class SynthSample:
    image: np.ndarray
    text: str
    boxes: List[Tuple[int, int, int, int, int]]  # x1,y1,x2,y2,class_id


def _augment_digit(digit_img: np.ndarray) -> np.ndarray:
    img = digit_img.copy()
    img = geom_aug(image=img)['image']
    return img


def generate_synthetic_sample(min_digits=2, max_digits=5, canvas_h=64) -> SynthSample:
    n = random.randint(min_digits, max_digits)
    digit_classes = [random.randint(0, 9) for _ in range(n)]

    digit_imgs = []
    for cls in digit_classes:
        src = by_digit[cls][np.random.randint(0, len(by_digit[cls]))]
        aug = _augment_digit(src)
        aug = cv2.resize(aug, (28, 28), interpolation=cv2.INTER_AREA)
        digit_imgs.append(aug)

    cursor_x = 6
    boxes = []
    parts = []

    # Pre-compute final width with random spacing/overlap.
    for i in range(n):
        parts.append((digit_classes[i], digit_imgs[i]))

    est_w = 12
    for i in range(n):
        est_w += 28
        if i < n - 1:
            gap = random.randint(0, 5)
            overlap = random.randint(0, 3)  # <= ~10% of 28
            est_w += gap - overlap
    est_w += 12

    canvas = np.zeros((canvas_h, max(est_w, 70)), dtype=np.uint8)

    for i, (cls, dimg) in enumerate(parts):
        y = random.randint(8, canvas_h - 28 - 8)

        mask = dimg > 25
        x1, y1 = cursor_x, y
        x2, y2 = cursor_x + 28, y + 28

        roi = canvas[y1:y2, x1:x2]
        roi[mask] = np.maximum(roi[mask], dimg[mask])
        canvas[y1:y2, x1:x2] = roi

        boxes.append((x1, y1, x2, y2, cls))

        if i < n - 1:
            gap = random.randint(0, 5)
            overlap = random.randint(0, 3)
            cursor_x += 28 + gap - overlap

    canvas = photo_aug(image=canvas)['image']
    text = ''.join(str(x) for x in digit_classes)
    return SynthSample(canvas, text, boxes)

In [ ]:
# Generate dataset (10k+ train samples as requested)
N_SAMPLES = 12500
samples = [generate_synthetic_sample() for _ in range(N_SAMPLES)]

print('Generated samples:', len(samples))

fig, axes = plt.subplots(2, 4, figsize=(14, 5))
for ax, s in zip(axes.flat, random.sample(samples, 8)):
    ax.imshow(s.image, cmap='gray')
    ax.set_title(s.text)
    for (x1, y1, x2, y2, c) in s.boxes:
        ax.add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor='lime', linewidth=1))
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
idx = np.arange(len(samples))
train_idx, val_idx = train_test_split(idx, test_size=0.2, random_state=SEED)
train_samples = [samples[i] for i in train_idx]
val_samples = [samples[i] for i in val_idx]

print('Train:', len(train_samples), 'Val:', len(val_samples))

## 4) Variant 1: Segmentation + single-digit CNN classifier

Pipeline:
1. Train a high-accuracy single-digit CNN on original 28x28 digits.
2. Segment each sequence image using thresholding + contours.
3. If regions are too wide (touching digits), apply projection-profile split.
4. Classify each crop and concatenate predictions.

In [ ]:
def build_digit_cnn_tf(input_shape=(28, 28, 1), n_classes=10):
    inp = keras.Input(shape=input_shape)
    x = layers.Conv2D(32, 3, padding='same', activation='relu')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(32, 3, padding='same', activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    out = layers.Dense(n_classes, activation='softmax')(x)

    model = keras.Model(inp, out)
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

In [ ]:
# Train/val split for single-digit classifier (TensorFlow)
x_train, x_val, y_train, y_val = train_test_split(
    images, labels, test_size=0.1, random_state=SEED, stratify=labels
)

x_train = (x_train.astype(np.float32) / 255.0)[..., None]
x_val = (x_val.astype(np.float32) / 255.0)[..., None]

model = build_digit_cnn_tf()
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint('single_digit_cnn_tf_best.keras', monitor='val_accuracy', save_best_only=True)
]

hist = model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=15,
    batch_size=128,
    callbacks=callbacks,
    verbose=1
)

print('Best val acc:', float(np.max(hist.history['val_accuracy'])))

In [ ]:
# Segmentation helpers for sequence images

def split_wide_box(bin_img: np.ndarray, x1: int, y1: int, x2: int, y2: int) -> List[Tuple[int, int, int, int]]:
    crop = bin_img[y1:y2, x1:x2]
    proj = crop.sum(axis=0)
    proj = cv2.GaussianBlur(proj.astype(np.float32), (5, 1), 0)

    minima = []
    for i in range(2, len(proj)-2):
        if proj[i] < proj[i-1] and proj[i] < proj[i+1] and proj[i] < np.percentile(proj, 35):
            minima.append(i)

    if len(minima) == 0:
        return [(x1, y1, x2, y2)]

    cuts = [0] + minima + [crop.shape[1]-1]
    out = []
    for a, b in zip(cuts[:-1], cuts[1:]):
        if b - a < 8:
            continue
        out.append((x1 + a, y1, x1 + b, y2))
    return out if out else [(x1, y1, x2, y2)]


def segment_digits_opencv(img: np.ndarray) -> List[Tuple[int, int, int, int]]:
    blur = cv2.GaussianBlur(img, (3, 3), 0)
    _, th = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel = np.ones((2, 2), np.uint8)
    th = cv2.erode(th, kernel, iterations=1)

    contours, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boxes = []
    for c in contours:
        x, y, w, h = cv2.boundingRect(c)
        if w * h < 30:
            continue
        boxes.append((x, y, x+w, y+h))

    boxes = sorted(boxes, key=lambda b: b[0])
    refined = []
    for (x1, y1, x2, y2) in boxes:
        w = x2 - x1
        h = y2 - y1
        if w > int(1.35 * h):
            refined.extend(split_wide_box(th, x1, y1, x2, y2))
        else:
            refined.append((x1, y1, x2, y2))

    return sorted(refined, key=lambda b: b[0])


def classify_segments(img: np.ndarray, boxes: List[Tuple[int, int, int, int]], model) -> str:
    preds = []
    for (x1, y1, x2, y2) in boxes:
        crop = img[max(0, y1-1):min(img.shape[0], y2+1), max(0, x1-1):min(img.shape[1], x2+1)]
        crop = cv2.resize(crop, (28, 28), interpolation=cv2.INTER_AREA)
        x = (crop.astype(np.float32)/255.0)[None, :, :, None]
        p = int(np.argmax(model.predict(x, verbose=0), axis=1)[0])
        preds.append(str(p))
    return ''.join(preds)

In [ ]:
# Evaluate segmentation + classifier on synthetic validation set
model = keras.models.load_model('single_digit_cnn_tf_best.keras')

def evaluate_segmentation_pipeline(val_samples, max_eval=1200):
    subset = val_samples[:max_eval]
    seq_ok, total = 0, 0
    digit_ok, digit_total = 0, 0

    for s in subset:
        boxes = segment_digits_opencv(s.image)
        pred = classify_segments(s.image, boxes, model)
        gt = s.text

        seq_ok += int(pred == gt)
        total += 1

        m = min(len(pred), len(gt))
        digit_ok += sum(int(pred[i] == gt[i]) for i in range(m))
        digit_total += max(len(gt), 1)

    return {
        'sequence_acc': seq_ok / total,
        'digit_acc': digit_ok / digit_total,
        'n_eval': total
    }

seg_metrics = evaluate_segmentation_pipeline(val_samples)
print(seg_metrics)

## 5) Variant 2: YOLOv8 end-to-end detector (recommended)

We convert synthetic data into YOLO format:
- image file: `.png`
- label file: one row per digit -> `class x_center y_center width height` (normalized)

Then we fine-tune `yolov8n.pt` for digit detection/classification on sequence images.

In [ ]:
# Prepare YOLO directory structure
YOLO_ROOT = Path('/kaggle/working/yolo_digits')
if YOLO_ROOT.exists():
    shutil.rmtree(YOLO_ROOT)

for split in ['train', 'val']:
    (YOLO_ROOT / 'images' / split).mkdir(parents=True, exist_ok=True)
    (YOLO_ROOT / 'labels' / split).mkdir(parents=True, exist_ok=True)


def save_yolo_sample(sample: SynthSample, out_img_path: Path, out_lbl_path: Path):
    img = sample.image
    h, w = img.shape
    cv2.imwrite(str(out_img_path), img)

    lines = []
    for (x1, y1, x2, y2, cls) in sample.boxes:
        xc = ((x1 + x2) / 2) / w
        yc = ((y1 + y2) / 2) / h
        bw = (x2 - x1) / w
        bh = (y2 - y1) / h
        lines.append(f"{cls} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")

    out_lbl_path.write_text('\n'.join(lines))

for i, s in enumerate(train_samples):
    save_yolo_sample(s, YOLO_ROOT / 'images/train' / f'{i:05d}.png', YOLO_ROOT / 'labels/train' / f'{i:05d}.txt')

for i, s in enumerate(val_samples):
    save_yolo_sample(s, YOLO_ROOT / 'images/val' / f'{i:05d}.png', YOLO_ROOT / 'labels/val' / f'{i:05d}.txt')

names = {i: str(i) for i in range(10)}
data_yaml = YOLO_ROOT / 'digits.yaml'
data_yaml.write_text(
    'path: ' + str(YOLO_ROOT) + '\n'
    + 'train: images/train\n'
    + 'val: images/val\n'
    + 'names:\n'
    + '\n'.join([f'  {k}: "{v}"' for k, v in names.items()]) + '\n'
)

print('YOLO dataset prepared at', YOLO_ROOT)

In [ ]:
from ultralytics import YOLO

# Train YOLOv8n (10-20 epochs suggested)
yolo = YOLO('yolov8n.pt')
results = yolo.train(
    data=str(data_yaml),
    epochs=15,
    imgsz=416,
    batch=32,
    project='/kaggle/working',
    name='yolo_digits_exp',
    pretrained=True,
    optimizer='Adam',
    lr0=1e-3,
    patience=5,
    device=0 if torch.cuda.is_available() else 'cpu'
)

In [ ]:
# Validate YOLO model
metrics = yolo.val(data=str(data_yaml), imgsz=416, split='val')
print(metrics)

In [ ]:
# Inference utility: sort detections left->right and return sequence string

best_weights = Path('/kaggle/working/yolo_digits_exp/weights/best.pt')
best_yolo = YOLO(str(best_weights))

def yolo_predict_string(img: np.ndarray, conf=0.25):
    pred = best_yolo.predict(source=img, conf=conf, verbose=False)[0]

    boxes = pred.boxes.xyxy.cpu().numpy() if pred.boxes is not None else np.zeros((0, 4))
    cls = pred.boxes.cls.cpu().numpy().astype(int) if pred.boxes is not None else np.array([], dtype=int)

    order = np.argsort(boxes[:, 0]) if len(boxes) else []
    chars = [str(cls[i]) for i in order]
    seq = ''.join(chars)

    vis = cv2.cvtColor(img.copy(), cv2.COLOR_GRAY2BGR)
    for i in order:
        x1, y1, x2, y2 = boxes[i].astype(int)
        c = int(cls[i])
        cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 1)
        cv2.putText(vis, str(c), (x1, max(10, y1-3)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)
    return seq, vis

In [ ]:
# Sequence accuracy for YOLO on val subset

def eval_yolo_sequence_accuracy(samples, n_eval=1000):
    subset = samples[:n_eval]
    ok = 0
    for s in subset:
        pred, _ = yolo_predict_string(s.image, conf=0.2)
        ok += int(pred == s.text)
    return ok / len(subset)

yolo_seq_acc = eval_yolo_sequence_accuracy(val_samples, n_eval=800)
print('YOLO sequence accuracy:', yolo_seq_acc)

## 6) Side-by-side comparison and error analysis

In [ ]:
# Visual compare
n_show = 8
fig, axes = plt.subplots(n_show, 2, figsize=(12, 3*n_show))

for r, s in enumerate(random.sample(val_samples, n_show)):
    # Variant 1
    seg_boxes = segment_digits_opencv(s.image)
    seg_pred = classify_segments(s.image, seg_boxes, model)
    v1 = cv2.cvtColor(s.image.copy(), cv2.COLOR_GRAY2BGR)
    for (x1, y1, x2, y2) in seg_boxes:
        cv2.rectangle(v1, (x1, y1), (x2, y2), (255, 255, 0), 1)

    # Variant 2
    yolo_pred, yolo_vis = yolo_predict_string(s.image)

    axes[r, 0].imshow(v1)
    axes[r, 0].set_title(f"Seg+CNN | GT={s.text} Pred={seg_pred}")
    axes[r, 0].axis('off')

    axes[r, 1].imshow(yolo_vis)
    axes[r, 1].set_title(f"YOLOv8 | GT={s.text} Pred={yolo_pred}")
    axes[r, 1].axis('off')

plt.tight_layout()
plt.show()

## 7) Inference on custom images (upload or external set)

You can load your own image (e.g., phone-captured handwriting), preprocess to grayscale, and run both models.

In [ ]:
# Example placeholder for custom inference
# custom = cv2.imread('/kaggle/input/your-dataset/custom.png', cv2.IMREAD_GRAYSCALE)
# custom = cv2.resize(custom, (416, 64))
# pred_yolo, vis = yolo_predict_string(custom)
# seg_boxes = segment_digits_opencv(custom)
# pred_seg = classify_segments(custom, seg_boxes, model)
# print('YOLO prediction:', pred_yolo)
# print('Seg+CNN prediction:', pred_seg)
# plt.figure(figsize=(12,3)); plt.imshow(vis); plt.axis('off'); plt.show()

## 8) Save/export models for deployment

- CNN classifier: `single_digit_cnn_tf_best.keras`
- YOLO detector: `/kaggle/working/yolo_digits_exp/weights/best.pt`

These can be used in a Streamlit or FastAPI app by loading the weights and running `yolo_predict_string`.

In [ ]:
model.save('/kaggle/working/single_digit_cnn_tf_best.keras')
print('Saved:', '/kaggle/working/single_digit_cnn_tf_best.keras')
print('Saved:', '/kaggle/working/yolo_digits_exp/weights/best.pt')

## 9) Live webcam inference cell (Kaggle)

After training, run the next cell to open a **live webcam demo** using Gradio.

- It captures frames from your webcam.
- Runs OpenCV segmentation + TensorFlow classifier in real time.
- Returns predicted digit string and a visualization with boxes.

> Note: In Kaggle, webcam/browser permissions must be granted in the notebook output panel.

In [ ]:
# Live webcam demo (Gradio) for real-time sequence prediction
# Uncomment install if needed:
# !pip install -q gradio

import gradio as gr


def predict_from_webcam(frame):
    if frame is None:
        return "", None

    # Convert RGB/BGR frame to grayscale
    if frame.ndim == 3:
        gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
    else:
        gray = frame.copy()

    boxes = segment_digits_opencv(gray)
    pred = classify_segments(gray, boxes, model)

    vis = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
    for (x1, y1, x2, y2) in boxes:
        cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 255), 2)

    return pred, vis


demo = gr.Interface(
    fn=predict_from_webcam,
    inputs=gr.Image(sources=["webcam"], type="numpy", label="Webcam"),
    outputs=[
        gr.Textbox(label="Predicted digit sequence"),
        gr.Image(type="numpy", label="Segmented view")
    ],
    live=True,
    title="Live Handwritten Digit String Recognition"
)

demo.launch(share=False, debug=False)

## 10) Final summary (expected performance)

Typical outcomes after tuning on synthetic data:
- Single-digit CNN validation accuracy: **~99%+** on isolated digits.
- Segmentation+CNN sequence accuracy: **good on separated digits**, drops when heavy touching occurs.
- YOLOv8 sequence accuracy on synthetic touching digits: often **95%+** with enough overlap-rich training samples and balanced augmentations.

### Improvement ideas
- Add a **CRNN + CTC** model as a third variant for direct sequence transcription.
- Use curriculum: start with separated digits then increase overlap ratio each epoch.
- Mix in real handwritten string datasets (e.g., NIST SD19, custom captures) for domain adaptation.
- Apply test-time augmentations and weighted box fusion for stronger detection stability.